## Texas and Georgia Auction.com

In [158]:
import pandas as pd
from pathlib import Path
import numpy as np
from rapidfuzz import process, fuzz
import subprocess
import json
import os

In [159]:
folder = Path('raw_data/auction_com')

excel_files = list(folder.glob('*.xlsx'))

# Combine all texas
auction = pd.concat(
    [pd.read_excel(file, dtype={'Tax ID': 'string'}) for file in excel_files],
    ignore_index=True
)

/opt/homebrew/Caskroom/miniconda/base/envs/py38/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/homebrew/Caskroom/miniconda/base/envs/py38/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/homebrew/Caskroom/miniconda/base/envs/py38/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/homebrew/Caskroom/miniconda/base/envs/py38/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style,

In [160]:
print(f"Files combined: {len(excel_files)}")
print(f"Total rows: {len(auction)}")
print(f"Total columns: {len(auction.columns)}")

display(auction.head())

Files combined: 4
Total rows: 1946
Total columns: 19


,Listing Type,Property Address,Property City,County,State,Zip,Status,Auction Date,Property Type,Square Footage,Bedrooms,Baths,Lot Size,Year Built,Est. Market Value,Starting Bid,Est. Credit Bid,APN,Trustee Number
0,Foreclosure,1827 POSEIDON DRIVE,LANCASTER,Dallas,TX,75134.0,SALE_PENDING,2026-11-03,Single family home,2599.0,4.0,3.0,0.15,2006.0,327065.0,NaN,NaN,NaN,TS 10950319
1,Foreclosure,309 ADAMTOWNE DR,EL PASO,El Paso,TX,79928.0,SALE_PENDING,2026-11-03,Single family home,1932.0,4.0,3.0,0.12,2024.0,313422.0,NaN,NaN,NaN,TS 10936573
2,Foreclosure,3011 LAYNE ST,LA PORTE,Harris,TX,77571.0,SALE_PENDING,2026-11-03,Single family home,1322.0,3.0,2.0,0.11,2022.0,230043.0,NaN,NaN,NaN,26-000441-671
3,Foreclosure,305 HAYWOOD DR,BENBROOK,Tarrant,TX,76126.0,SALE_PENDING,2026-11-03,Single family home,1911.0,4.0,2.0,0.22,1980.0,334986.0,NaN,NaN,NaN,9926-3917
4,Foreclosure,204 KING WILLIAM DR,LA PORTE,Harris,TX,77571.0,SALE_PENDING,2026-11-03,Single family home,3341.0,4.0,2.5,0.15,2006.0,308511.0,NaN,NaN,126-056-001-0005,26-013461


In [161]:
#Create full address for record name

auction["Last Name"] = (
    auction["Property Address"].str.title() + ", " +
    auction["Property City"].str.title() + ", " +
    auction["State"].str.title() + " " +
    auction["Zip"].astype(str).str.replace(r"\.0$", "", regex=True)
)

## Match Addresses

In [162]:
query = """
SELECT
    Id,
    Name,
    Auction_Status__c,
    Auction_Trustee_Lookup__r.Name,
    Auction_Listing_ID__c,
    Auction_Opening_Bid__c
FROM Lead
WHERE (
    Auction_Date__c = LAST_N_DAYS:60
    OR Auction_Date__c = NEXT_N_DAYS:30)
AND State_Lookup__r.Name IN ('Texas', 'Georgia')
"""

result = subprocess.run(
    [
        "sf", "data", "query",
        "--query", query,
        "--target-org", "production_org",
        "--json"
    ],
    capture_output=True,
    text=True,
    env={**os.environ, "NO_COLOR": "1", "FORCE_COLOR": "0"}
)

if result.returncode != 0:
    print("Salesforce Error:")
    print(result.stderr)
    print(result.stdout[:1000])

else:
    # Remove ANSI color codes if any remain
    import re

    clean_output = re.sub(
        r'\x1b\[[0-9;]*m',
        '',
        result.stdout
    )

    data = json.loads(clean_output)

    sf_leads = pd.json_normalize(data['result']['records'])

    print(f"Successfully retrieved {len(sf_leads)} Salesforce records")
    display(sf_leads.head())

Successfully retrieved 5799 Salesforce records


,Id,Name,Auction_Status__c,Auction_Listing_ID__c,Auction_Opening_Bid__c,attributes.type,attributes.url,Auction_Trustee_Lookup__r.attributes.type,Auction_Trustee_Lookup__r.attributes.url,Auction_Trustee_Lookup__r.Name
0,00QQg00000nIBX3MAO,"209 Rosewood Drive, Brunswick, GA 31525",Killed,2183399,100755.64,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000n...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Auction.com
1,00QQg00000nIBX4MAO,"1604 Cave Spring Rd Sw, Rome, GA 30161",Killed,2182824,161200.00,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000n...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Auction.com
2,00QQg00000nIBX5MAO,"6851 Alexander Pkwy, Douglasville, GA 30135",Killed,2182090,224360.00,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000n...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Auction.com
3,00QQg00000nIBX6MAO,"4873 Old Belair Lane, Grovetown, GA 30813",Killed,2181984,NaN,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000n...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Auction.com
4,00QQg00000nIBX7MAO,"10475 Ga Highway 57, Mc Intyre, GA 31054",Killed,2181983,83840.00,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000n...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Auction.com


In [163]:
#Rename sf columns 

sf_leads = sf_leads.rename(columns={
    'Name': 'Last Name',
    'Auction_Status__c': 'Auction Status',
    'Auction_Trustee_Lookup__r.Name': 'Auction Trustee',
    'Auction_Listing_ID__c': 'Auction Listing ID',
    'Auction_Opening_Bid__c': 'Auction Opening Bid'
})

In [164]:
sf_leads[['Last Name', 'Auction Status', 'Auction Trustee',
          'Auction Listing ID', 'Auction Opening Bid']].head()

,Last Name,Auction Status,Auction Trustee,Auction Listing ID,Auction Opening Bid
0,"209 Rosewood Drive, Brunswick, GA 31525",Killed,Auction.com,2183399,100755.64
1,"1604 Cave Spring Rd Sw, Rome, GA 30161",Killed,Auction.com,2182824,161200.00
2,"6851 Alexander Pkwy, Douglasville, GA 30135",Killed,Auction.com,2182090,224360.00
3,"4873 Old Belair Lane, Grovetown, GA 30813",Killed,Auction.com,2181984,NaN
4,"10475 Ga Highway 57, Mc Intyre, GA 31054",Killed,Auction.com,2181983,83840.00


In [165]:
#Fuzzy Match

threshold = 85

# Normalize names
def normalize_name(value):
    if pd.isna(value):
        return ""
    return " ".join(str(value).upper().strip().split())

# Salesforce lookup
sf_lookup = {}

for name in sf_leads['Last Name'].dropna().unique():
    normalized = normalize_name(name)
    if normalized:
        sf_lookup.setdefault(normalized, name)

sf_names = list(sf_lookup.keys())

# Keep original Auction.com Last Name
auction['Auction Last Name'] = auction['Last Name']

# Fuzzy match
def fuzzy_match_name(name):
    normalized = normalize_name(name)

    if not normalized or not sf_names:
        return pd.Series([None, None])

    match = process.extractOne(
        normalized,
        sf_names,
        scorer=fuzz.ratio
    )

    if match:
        matched_name, score, _ = match
        return pd.Series([sf_lookup[matched_name], round(score, 2)])

    return pd.Series([None, None])

# Store Salesforce match and confidence
auction[['Salesforce Last Name', 'Match Confidence']] = (
    auction['Auction Last Name'].apply(fuzzy_match_name)
)

# Use Salesforce Last Name only if confidence >= 85
auction['Last Name'] = auction['Auction Last Name']

matched = auction['Match Confidence'] >= threshold

auction.loc[matched, 'Last Name'] = (
    auction.loc[matched, 'Salesforce Last Name']
)

In [166]:
#View matches

display(auction[
    ['Auction Last Name', 'Salesforce Last Name', 'Match Confidence']
])

,Auction Last Name,Salesforce Last Name,Match Confidence
0,"1827 Poseidon Drive, Lancaster, Tx 75134","927 Poinsettia Drive, Lancaster, TX 75146",83.95
1,"309 Adamtowne Dr, El Paso, Tx 79928","12238 Rathmore Dr, El Paso, TX 79928",78.87
2,"3011 Layne St, La Porte, Tx 77571","3135 Hamilton St, La Porte, TX 77571",81.16
3,"305 Haywood Dr, Benbrook, Tx 76126","7305 Hogan Dr, Benbrook, TX 76126",86.57
4,"204 King William Dr, La Porte, Tx 77571","704 Garden Walk Dr, La Porte, TX 77571",77.92
...,...,...,...
1941,"577 Everett Springs Rd Sw, Calhoun, Ga 30701","1604 Cave Spring Rd Sw, Rome, GA 30161",65.85
1942,"3916 Duchess Ct, Valdosta, Ga 31605","5003 Peachtree St, Valdosta, GA 31605",75.00
1943,"1621 Alcovy Ridge Crossing, Loganville, Ga 30052","702 Buckhorn Ct, Loganville, GA 30052",68.24
1944,NaN,None,NaN


In [167]:
#Look at differences

#Common street suffixes
suffix_map = {
    'DRIVE': 'DR',
    'STREET': 'ST',
    'AVENUE': 'AVE',
    'ROAD': 'RD',
    'LANE': 'LN',
    'BOULEVARD': 'BLVD',
    'COURT': 'CT',
    'CIRCLE': 'CIR',
    'PLACE': 'PL',
    'TERRACE': 'TER',
    'TRAIL': 'TRL',
    'PARKWAY': 'PKWY',
    'HIGHWAY': 'HWY',
    'WAY': 'WAY'
}

def get_address_parts(address):
    if pd.isna(address):
        return '', ''

    address = str(address).strip()
    parts = [p.strip() for p in address.split(',')]

    # Expected format: Street, City, TX ZIP
    street = parts[0] if len(parts) >= 1 else ''
    city = parts[1] if len(parts) >= 3 else ''

    return street.upper(), city.upper()

def normalize_street(street):
    street = re.sub(r'[^\w\s]', '', street.upper())
    words = street.split()
    return ' '.join(suffix_map.get(word, word) for word in words)

def compare_addresses(row):
    auction_street, auction_city = get_address_parts(row['Auction Last Name'])
    sf_street, sf_city = get_address_parts(row['Salesforce Last Name'])

    # Detect abbreviation-only differences
    suffix_difference = (
        auction_street != sf_street
        and normalize_street(auction_street) == normalize_street(sf_street)
    )

    city_difference = auction_city != sf_city

    return pd.Series({
        'Street Suffix Difference': suffix_difference,
        'City Difference': city_difference,
        'Auction City': auction_city,
        'Salesforce City': sf_city
    })

# Compare matched records
address_comparison = auction[
    (auction['Match Confidence'] >= 85) &
    auction['Salesforce Last Name'].notna()
].copy()

address_comparison = address_comparison.join(
    address_comparison.apply(compare_addresses, axis=1)
)

In [168]:
differences = address_comparison[
    address_comparison['Street Suffix Difference'] |
    address_comparison['City Difference']
]

display(differences[
    [
        'Last Name',
        'Auction Last Name',
        'Salesforce Last Name',
        'Match Confidence',
        'Street Suffix Difference',
        'City Difference'
    ]
])

,Last Name,Auction Last Name,Salesforce Last Name,Match Confidence,Street Suffix Difference,City Difference
126,"9450 Valley Way Dr, San Antonio, TX 78250","9450 Valley Way Drive, San Antonio, Tx 78250","9450 Valley Way Dr, San Antonio, TX 78250",96.47,True,False
187,"612 Bunker Hill Rd, Fate, TX 75189","612 Bunker Hill Road, Fate, Tx 75189","612 Bunker Hill Rd, Fate, TX 75189",97.14,True,False
382,"9607 Walnut St, Dallas, TX 75243","9607 Walnut Street, Dallas, Tx 75243","9607 Walnut St, Dallas, TX 75243",94.12,True,False
550,"7806 Garner Rd, Rowlett, TX 75088","7806 Garner Road, Rowlett, Tx 75088","7806 Garner Rd, Rowlett, TX 75088",97.06,True,False
564,"12610 Raven South Dr, Cypress, TX 77429","12610 Raven South Drive, Cypress, Tx 77429","12610 Raven South Dr, Cypress, TX 77429",96.30,True,False
646,"11720 Kurth Dr, Frisco, TX 75036","11720 Kurth Drive, Frisco, Tx 75034","11720 Kurth Dr, Frisco, TX 75036",92.54,True,False
648,"6742 Spring Hurst St, San Antonio, TX 78249","6742 Spring Hurst Street, San Antonio, Tx 78249","6742 Spring Hurst St, San Antonio, TX 78249",95.56,True,False
734,"1302 Harbor Ct, Lancaster, TX 75134","1302 Harbor Court, Lancaster, Tx 75134","1302 Harbor Ct, Lancaster, TX 75134",95.89,True,False
777,"9643 Towne Lake Pkwy, Cypress, TX 77433","9643 Towne Lake Parkway, Cypress, Tx 77433","9643 Towne Lake Pkwy, Cypress, TX 77433",96.30,True,False
781,"13222 Arden Ridge Ln, Houston, TX 77014","13222 Arden Ridge Lane, Houston, Tx 77014","13222 Arden Ridge Ln, Houston, TX 77014",97.50,True,False


In [169]:
print("Street suffix differences:",
      address_comparison['Street Suffix Difference'].sum())

print("City differences:",
      address_comparison['City Difference'].sum())

print("Both:",
      (
          address_comparison['Street Suffix Difference'] &
          address_comparison['City Difference']
      ).sum())

Street suffix differences: 18
City differences: 0
Both: 0


In [170]:
low_confidence = auction[
    (auction['Match Confidence'] < 85) |
    (auction['Match Confidence'].isna())
]

display(low_confidence[
    ['Last Name','Auction Last Name', 'Salesforce Last Name', 'Match Confidence']
])

,Last Name,Auction Last Name,Salesforce Last Name,Match Confidence
0,"1827 Poseidon Drive, Lancaster, Tx 75134","1827 Poseidon Drive, Lancaster, Tx 75134","927 Poinsettia Drive, Lancaster, TX 75146",83.95
1,"309 Adamtowne Dr, El Paso, Tx 79928","309 Adamtowne Dr, El Paso, Tx 79928","12238 Rathmore Dr, El Paso, TX 79928",78.87
2,"3011 Layne St, La Porte, Tx 77571","3011 Layne St, La Porte, Tx 77571","3135 Hamilton St, La Porte, TX 77571",81.16
4,"204 King William Dr, La Porte, Tx 77571","204 King William Dr, La Porte, Tx 77571","704 Garden Walk Dr, La Porte, TX 77571",77.92
5,"2732 Trophy Club Dr, Trophy Club, Tx 76262","2732 Trophy Club Dr, Trophy Club, Tx 76262","2823 Macquarie, Trophy Club, TX 76262",73.42
...,...,...,...,...
1941,"577 Everett Springs Rd Sw, Calhoun, Ga 30701","577 Everett Springs Rd Sw, Calhoun, Ga 30701","1604 Cave Spring Rd Sw, Rome, GA 30161",65.85
1942,"3916 Duchess Ct, Valdosta, Ga 31605","3916 Duchess Ct, Valdosta, Ga 31605","5003 Peachtree St, Valdosta, GA 31605",75.00
1943,"1621 Alcovy Ridge Crossing, Loganville, Ga 30052","1621 Alcovy Ridge Crossing, Loganville, Ga 30052","702 Buckhorn Ct, Loganville, GA 30052",68.24
1944,NaN,NaN,None,NaN


## Format Data

In [171]:
#Online Bidding
auction['Online Bidding'] = 'Yes'

#Auction Trustee
auction['Auction Trustee (Lookup)'] = 'Auction.com'

In [172]:
texascounties = pd.read_csv('sf_data/texascounties.csv')
georgiacounties = pd.read_csv('sf_data/georgiacounties.csv')
leadmanagers = pd.read_csv('sf_data/leadmanagers.csv')

#Rename original County
auction = auction.rename(columns={'County': 'oldCounty'})

#Create county lookups
county_lookup = {}

for state, df in [('TX', texascounties), ('GA', georgiacounties)]:
    for _, row in df.iterrows():
        county_lookup[(state, row['County'])] = {
            'Market': row['Market'],
            'Lead Manager Name': row['Lead Manager']
        }

#Find Market and Lead Manager Name
def find_county_details(row):
    result = county_lookup.get(
        (row['State'], row['oldCounty']), {}
    )

    return pd.Series({
        'Market': result.get('Market', ''),
        'Lead Manager Name': result.get('Lead Manager Name', '')
    })

auction[['Market', 'Lead Manager Name']] = auction.apply(
    find_county_details, axis=1
)

#Dallas County Online override
auction.loc[
    (auction['State'] == 'TX') &
    (auction['oldCounty'] == 'Dallas'),
    'Lead Manager Name'
] = 'Ernie Loreto'

#Find Lead Manager Salesforce ID
manager_lookup = (
    leadmanagers.drop_duplicates(subset='Lead Manager')
    .set_index('Lead Manager')['Salesforce ID']
)

auction['Lead Manager'] = auction['Lead Manager Name'].map(
    manager_lookup
)

#Create County column
auction['County'] = (
    auction['oldCounty'] + ' - ' + auction['State']
)

#Create Decision column
auction['Decision'] = 'Add New'

auction.loc[
    auction['Match Confidence'] >= 85,
    'Decision'
] = 'Update'

auction.loc[
    auction['Market'] == 'Exclude',
    'Decision'
] = 'Exclude'

In [173]:
auction[['oldCounty', 'County', 'Market', 'Lead Manager Name']].head()

,oldCounty,County,Market,Lead Manager Name
0,Dallas,Dallas - TX,Dallas,Ernie Loreto
1,El Paso,El Paso - TX,TX Small County,Kevin Rapport
2,Harris,Harris - TX,Houston,Michael Shiner
3,Tarrant,Tarrant - TX,Dallas,Ernie Loreto
4,Harris,Harris - TX,Houston,Michael Shiner


In [174]:
display(auction.loc[
    auction['Market'] == 'Exclude',
    ['Last Name', 'State', 'County', 'Market', 'Lead Manager Name', 'Decision']
])

,Last Name,State,County,Market,Lead Manager Name,Decision
8,"7031 Krause Dr, Missouri City, Tx 77489",TX,Fort Bend - TX,Exclude,NaN,Exclude
32,"14869 Prickly Pear Cove, Magnolia, Tx 77357",TX,Montgomery - TX,Exclude,NaN,Exclude
68,"22355 Dominguez Drive, Porter, Tx 77365",TX,Montgomery - TX,Exclude,NaN,Exclude
73,"27022 Calaway Falls Lane, Katy, Tx 77494",TX,Fort Bend - TX,Exclude,NaN,Exclude
81,"13736 Bighorn Trail, Willis, Tx 77378",TX,Montgomery - TX,Exclude,NaN,Exclude
...,...,...,...,...,...,...
1931,"673 Corsica Ln, Stockbridge, Ga 30281",GA,Henry - GA,Exclude,NaN,Exclude
1932,"145 Firewood St, Athens, Ga 30605",GA,Clarke - GA,Exclude,NaN,Exclude
1935,"6764 Columbus Rd, Lizella, Ga 31052",GA,Bibb - GA,Exclude,NaN,Exclude
1939,"920 Brookmere Ct, Atlanta, Ga 30349",GA,Fulton - GA,Exclude,NaN,Exclude


In [175]:
# Remove duplicate properties based on Last Name
auction = auction.drop_duplicates(
    subset=['Last Name'],
    keep='first'
).reset_index(drop=True)

In [176]:
before = len(auction)

auction = auction.drop_duplicates(
    subset=['Last Name'],
    keep='first'
).reset_index(drop=True)

print(f"Duplicates removed: {before - len(auction)}")
print(f"Remaining properties: {len(auction)}")

Duplicates removed: 0
Remaining properties: 1935


## Split Data

In [177]:
export_data = Path('export_data')

# Export records marked Update
auction[auction['Decision'] == 'Update'].to_csv(
    export_data / 'update_auction.csv',
    index=False
)

# Export records marked Add New
auction[auction['Decision'] == 'Add New'].to_csv(
    export_data / 'create_auction.csv',
    index=False
)

# Confirm exported records
print("Update Auction:", (auction['Decision'] == 'Update').sum())
print("Create Auction:", (auction['Decision'] == 'Add New').sum())
print("Excluded:", (auction['Decision'] == 'Exclude').sum())

Update Auction: 218
Create Auction: 1347
Excluded: 370


## Format update_auction

In [178]:
update_auction = auction[auction['Decision'] == 'Update'].copy()

#Add OB from Starting Bid
update_auction['Auction Opening Bid'] = (
    update_auction['Starting Bid'].replace(r'^\s*$', pd.NA, regex=True)
    .fillna('')
)

In [179]:
# Get Auction Status from Salesforce
sf_status_lookup = sf_leads.set_index('Last Name')['Auction Status']

update_auction['Salesforce Auction Status'] = (
    update_auction['Salesforce Last Name'].map(sf_status_lookup)
)

Priority,	Condition,	                                Auction Status

1	        Original Status = CANCELED,	                Cancelled

2	        Salesforce Auction Status = Killed,	        Killed

3	        Salesforce Auction Status = Clear to Bid,	Clear to Bid

4	        Auction Opening Bid has a value,	         Active

5	        Everything else,	                         Pending

In [180]:
# Default status
update_auction['Auction Status'] = 'Pending'

# Active if Opening Bid exists
has_bid = (
    update_auction['Auction Opening Bid'].notna() &
    update_auction['Auction Opening Bid'].astype(str).str.strip().ne('')
)

update_auction.loc[has_bid, 'Auction Status'] = 'Active'

# Preserve Salesforce Clear to Bid
#update_auction.loc[
    #update_auction['Salesforce Auction Status'] == 'Clear to Bid',
    #'Auction Status'
#] = 'Clear to Bid'

#Property type
update_auction.loc[
    update_auction['Property Type'] != 'Single family home',
    'Auction Status'
] = 'Killed'

# Canceled auction takes highest priority
update_auction.loc[
    update_auction['Status'] == 'CANCELED',
    'Auction Status'
] = 'Cancelled'

In [181]:
display(update_auction[
    [
        'Last Name',
        'Decision',
        'Status',
        'Salesforce Auction Status',
        'Starting Bid',
        'Auction Opening Bid',
        'Auction Status'
    ]
].head(20))

print(update_auction['Auction Status'].value_counts())


,Last Name,Decision,Status,Salesforce Auction Status,Starting Bid,Auction Opening Bid,Auction Status
3,"7305 Hogan Dr, Benbrook, TX 76126",Update,SALE_PENDING,Cancelled,NaN,,Pending
10,"5913 Lake Hubbard Pkwy Bldg: B Suite: 121, Gar...",Update,SALE_PENDING,Killed,NaN,,Killed
20,"3125 Millet Ave, Fort Worth, TX 76105",Update,SALE_PENDING,Cancelled,NaN,,Pending
35,"1515 Kew Garden Dr, Houston, TX 77047",Update,SALE_PENDING,Cancelled,201400.0,201400.0,Active
39,"3242 Lasses Blvd, San Antonio, TX 78223",Update,SALE_PENDING,Killed,NaN,,Pending
51,"406 College, Crandall, TX 75114",Update,SALE_PENDING,Pending,NaN,,Pending
53,"8514 Grosbeak Cove, San Antonio, TX 78245",Update,SALE_PENDING,Cancelled,NaN,,Pending
59,"505 Greenway Dr, Saginaw, TX 76179",Update,CANCELED,Cancelled,NaN,,Cancelled
69,"16911 Lagos Ln, Crosby, TX 77532",Update,SALE_PENDING,Cancelled,NaN,,Pending
71,"2820 Greene St, Celina, TX 75009",Update,SALE_PENDING,Cancelled,NaN,,Pending


Auction Status
Pending      159
Cancelled     40
Active        11
Killed         8
Name: count, dtype: int64


In [182]:
display(update_auction.loc[
    (update_auction['Auction Status'] == 'Killed') &
    (update_auction['Property Type'] != 'Single family home')
])

,Listing Type,Property Address,Property City,oldCounty,State,Zip,Status,Auction Date,Property Type,Square Footage,...,Online Bidding,Auction Trustee (Lookup),Market,Lead Manager Name,Lead Manager,County,Decision,Auction Opening Bid,Salesforce Auction Status,Auction Status
10,Foreclosure,5913 LAKE HUBBARD PKWY 121,GARLAND,Dallas,TX,75043.0,SALE_PENDING,2026-11-03,Condominium unit,920.0,...,Yes,Auction.com,Dallas,Ernie Loreto,005Qg00000UWrpV,Dallas - TX,Update,,Killed,Killed
119,Foreclosure,5102 CHENEVERT,HOUSTON,Harris,TX,77004.0,SALE_PENDING,2026-11-03,Duplex building,2472.0,...,Yes,Auction.com,Houston,Michael Shiner,005Qg00000Nsy1x,Harris - TX,Update,,Killed,Killed
216,Foreclosure,701 CIRCLE WAY,JARRELL,Williamson,TX,76537.0,SALE_PENDING,2026-11-03,Condominium unit,2520.0,...,Yes,Auction.com,Austin,Aaron Amuchastegui,005Qg00000Voakn,Williamson - TX,Update,,Killed,Killed
382,Foreclosure,9607 WALNUT STREET,DALLAS,Dallas,TX,75243.0,SALE_PENDING,2026-11-03,Condominium unit,1044.0,...,Yes,Auction.com,Dallas,Ernie Loreto,005Qg00000UWrpV,Dallas - TX,Update,,Killed,Killed
406,Foreclosure,14696 STUART ROAD,SAN ANTONIO,Bexar,TX,78223.0,SALE_PENDING,2026-11-03,Land,0.0,...,Yes,Auction.com,San Antonio,Mary Mokhtarian,005Qg00000V4EvN,Bexar - TX,Update,,Killed,Killed
1154,Foreclosure,1050 SCOTT RD,WEATHERFORD,Parker,TX,76088.0,SALE_PENDING,2026-11-03,Unknown,NaN,...,Yes,Auction.com,TX Small County,Kevin Rapport,005Qg00000VKIwb,Parker - TX,Update,,Pending,Killed
1207,Foreclosure,1919 CR 2130,"TELEPHONE,",Fannin,TX,75488.0,SALE_PENDING,2026-11-03,Unknown,NaN,...,Yes,Auction.com,TX Small County,Kevin Rapport,005Qg00000VKIwb,Fannin - TX,Update,,Cancelled,Killed
1773,Foreclosure,909 HIGHWAY 314,FAYETTEVILLE,Fayette,GA,30214,SALE_PENDING,2026-11-03,Unknown,NaN,...,Yes,Auction.com,West Georgia,David Riehle,0051H00000ASDES,Fayette - GA,Update,,Pending,Killed


In [ ]:
#FIRST IMPORT ONLY

#Format Auction Date Updated as MM/DD/YYYY
update_auction['Auction Date Updated'] = (
    pd.to_datetime(update_auction['Auction Date'], errors='coerce')
    .dt.strftime('%m/%d/%Y')
    .fillna('')
)

update_auction = update_auction.rename(columns={
    'Status': 'notStatus'
    'Year Built': 'notYearBuilt',
    'Baths': 'notBaths',
    'Trustee Number': 'NS Trustee Sale Number',
    'Market': 'notMarket',
    'County':'notCounty'
})

In [184]:
#ADDITIONAL IMPORTS
#Rename columns

#update_auction = update_auction.rename(columns={
    #'Status': 'notStatus',
    #'Year Built': 'notYearBuilt',
    #'Baths': 'notBaths',
    #'Trustee Number': 'NS Trustee Sale Number',
    #'Market': 'notMarket',
    #'County':'notCounty'
    #'Lead Manager': 'notLeadManager'
#})

In [185]:
update_auction.to_csv(
    export_data / 'update_auction.csv',
    index=False
)

## Format create_auction

In [202]:
create_auction = auction[auction['Decision'] == 'Add New'].copy()

In [203]:
#Format Tax ID

create_auction['Tax ID'] = (
    create_auction['APN']
    .astype('string')
    .replace(r'(?i)^\s*(null|nan|none)?\s*$', '', regex=True)
    .fillna('')
)

create_auction = create_auction.rename(columns={
    'APN': 'notAPN'
})

In [204]:
#Create unique identifier

market_map = {
    'TX Small County': 'TXSMALLCOUNTY',
    'Austin': 'AUSTIN',
    'Waco': 'WACO',
    'Dallas': 'DAL',
    'San Antonio': 'SAN_ANT',
    'Houston': 'HOUS'
}

create_auction['APN'] = (
    create_auction['Market'].map(market_map)
    + '_' + create_auction['Property Address'].astype('string').str.strip().str.replace(r'\s+', '_', regex=True)
    + '_' + create_auction['Zip'].astype('string').str.replace(r'\.0$', '', regex=True)
)

In [205]:
#Type of Property
create_auction['Type of Property'] = (
    create_auction['Property Type'].replace({
        'Single family home': 'SFR'
    })
)

#Auction Opening Bid
create_auction['Auction Opening Bid'] = (
    create_auction['Starting Bid']
    .replace(r'^\s*$', pd.NA, regex=True)
    .fillna('')
)

#Auction Status
create_auction['Auction Status'] = 'Pending'

# Active if opening bid exists
has_bid = (
    create_auction['Auction Opening Bid'].notna() &
    create_auction['Auction Opening Bid'].astype(str).str.strip().ne('')
)

create_auction.loc[
    has_bid,
    'Auction Status'
] = 'Active'

# Killed if not SFR (highest priority)
create_auction.loc[
    create_auction['Type of Property'] != 'SFR',
    'Auction Status'
] = 'Killed'

#Auction Date Updated - MM/DD/YYYY
create_auction['Auction Date Updated'] = (
    pd.to_datetime(create_auction['Auction Date'], errors='coerce')
    .dt.strftime('%m/%d/%Y')
    .fillna('')
)

#Rename columns
create_auction = create_auction.rename(columns={
    'Est. Market Value': 'RPR',
    'Bedrooms': 'Beds',
    'Trustee Number': 'NS Trustee Sale Number',
    ' Square Footage': 'Bldg Sq Ft',
    'Status': 'notStatus'
})

In [ ]:
# Remove trailing .0 from numeric columns
for col in ['Auction Opening Bid', 'RPR', 'Beds', 'Bldg Sq Ft', 'Baths']:
    create_auction[col] = (
        create_auction[col]
        .astype('string')
        .str.replace(r'\.0$', '', regex=True)
        .fillna('')
    )

In [210]:
display(create_auction[
    [
        'Last Name',
        'Market',
        'County',
        'State',
        'APN',
        'Tax ID',
        'notAPN',
        'Type of Property',
        'Auction Opening Bid',
        'Auction Status',
        'Auction Date Updated',
        'RPR',
        'Beds',
        'Baths',
        'NS Trustee Sale Number',
        'Bldg Sq Ft',
        'notStatus'
    ]
].head(20))

print(create_auction['Auction Status'].value_counts())

,Last Name,Market,County,State,APN,Tax ID,notAPN,Type of Property,Auction Opening Bid,Auction Status,Auction Date Updated,RPR,Beds,Baths,NS Trustee Sale Number,Bldg Sq Ft,notStatus
0,"1827 Poseidon Drive, Lancaster, Tx 75134",Dallas,Dallas - TX,TX,DAL_1827_POSEIDON_DRIVE_75134,,NaN,SFR,,Pending,11/03/2026,327065,4,3.0,TS 10950319,2599,SALE_PENDING
1,"309 Adamtowne Dr, El Paso, Tx 79928",TX Small County,El Paso - TX,TX,TXSMALLCOUNTY_309_ADAMTOWNE_DR_79928,,NaN,SFR,,Pending,11/03/2026,313422,4,3.0,TS 10936573,1932,SALE_PENDING
2,"3011 Layne St, La Porte, Tx 77571",Houston,Harris - TX,TX,HOUS_3011_LAYNE_ST_77571,,NaN,SFR,,Pending,11/03/2026,230043,3,2.0,26-000441-671,1322,SALE_PENDING
4,"204 King William Dr, La Porte, Tx 77571",Houston,Harris - TX,TX,HOUS_204_KING_WILLIAM_DR_77571,126-056-001-0005,126-056-001-0005,SFR,,Pending,11/03/2026,308511,4,2.5,26-013461,3341,SALE_PENDING
5,"2732 Trophy Club Dr, Trophy Club, Tx 76262",Dallas,Denton - TX,TX,DAL_2732_TROPHY_CLUB_DR_76262,536582DEN,536582DEN,SFR,,Pending,11/03/2026,931571,4,3.5,26-012531,3562,SALE_PENDING
6,"20539 Avery Grove Ct, Cypress, Tx 77433",Houston,Harris - TX,TX,HOUS_20539_AVERY_GROVE_CT_77433,,NaN,SFR,,Pending,11/03/2026,315220,4,2.5,9636-1879,2266,SALE_PENDING
7,"801 N D St, Midland, Tx 79701",TX Small County,Midland - TX,TX,TXSMALLCOUNTY_801_N_D_ST_79701,,NaN,SFR,,Pending,11/03/2026,,2,1.0,9658-3635,1425,SALE_PENDING
9,"507 Longworth Blvd, Duncanville, Tx 75116",Dallas,Dallas - TX,TX,DAL_507_LONGWORTH_BLVD_75116,,NaN,SFR,,Pending,11/03/2026,316849,3,2.0,9543-4066,1469,SALE_PENDING
11,"500 Dempsey Dr, Mckinney, Tx 75071",Dallas,Collin - TX,TX,DAL_500_DEMPSEY_DR_75071,R1302400R02001,R1302400R02001,SFR,,Pending,11/03/2026,343165,4,3.0,26-013607,2077,SALE_PENDING
12,"1914 Cross Draw Trail, Leander, Tx 78641",Austin,Williamson - TX,TX,AUSTIN_1914_CROSS_DRAW_TRAIL_78641,R483424,R483424,SFR,,Pending,11/03/2026,552997,4,3.0,25-016375-2,2866,SALE_PENDING


Auction Status
Pending    1216
Killed       85
Active       46
Name: count, dtype: int64


In [211]:
create_auction.head(5)


,Listing Type,Property Address,Property City,oldCounty,State,Zip,notStatus,Auction Date,Property Type,Bldg Sq Ft,...,Lead Manager Name,Lead Manager,County,Decision,Tax ID,APN,Type of Property,Auction Opening Bid,Auction Status,Auction Date Updated
0,Foreclosure,1827 POSEIDON DRIVE,LANCASTER,Dallas,TX,75134.0,SALE_PENDING,2026-11-03,Single family home,2599,...,Ernie Loreto,005Qg00000UWrpV,Dallas - TX,Add New,,DAL_1827_POSEIDON_DRIVE_75134,SFR,,Pending,11/03/2026
1,Foreclosure,309 ADAMTOWNE DR,EL PASO,El Paso,TX,79928.0,SALE_PENDING,2026-11-03,Single family home,1932,...,Kevin Rapport,005Qg00000VKIwb,El Paso - TX,Add New,,TXSMALLCOUNTY_309_ADAMTOWNE_DR_79928,SFR,,Pending,11/03/2026
2,Foreclosure,3011 LAYNE ST,LA PORTE,Harris,TX,77571.0,SALE_PENDING,2026-11-03,Single family home,1322,...,Michael Shiner,005Qg00000Nsy1x,Harris - TX,Add New,,HOUS_3011_LAYNE_ST_77571,SFR,,Pending,11/03/2026
4,Foreclosure,204 KING WILLIAM DR,LA PORTE,Harris,TX,77571.0,SALE_PENDING,2026-11-03,Single family home,3341,...,Michael Shiner,005Qg00000Nsy1x,Harris - TX,Add New,126-056-001-0005,HOUS_204_KING_WILLIAM_DR_77571,SFR,,Pending,11/03/2026
5,Foreclosure,2732 TROPHY CLUB DR,TROPHY CLUB,Denton,TX,76262.0,SALE_PENDING,2026-11-03,Single family home,3562,...,Ernie Loreto,005Qg00000UWrpV,Denton - TX,Add New,536582DEN,DAL_2732_TROPHY_CLUB_DR_76262,SFR,,Pending,11/03/2026


In [212]:
create_auction.to_csv(
    export_data / 'create_auction.csv',
    index=False
)